# Lab 6.4 — Half-life

**Goal:** simulate radioactive decay with random numbers, compare it with the exponential law, measure a half-life from your own data, and date some samples.

Every nucleus decays at random. In a short time step $\Delta t$, each surviving nucleus decays with probability

$$p = 1 - 2^{-\Delta t / T_{1/2}}$$

and the expected number left after time $t$ is

$$N = N_0 \left(\tfrac{1}{2}\right)^{t/T_{1/2}} = N_0\, e^{-\lambda t}, \qquad \lambda = \frac{\ln 2}{T_{1/2}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(seed=7)

## 1. Simulate decay

`simulate(N0, half_life, dt, steps)` returns the number of nuclei left after each step. Each step, every surviving nucleus 'tosses a coin' weighted by $p$.

In [ ]:
def simulate(N0, half_life, dt, steps):
    p = 1 - 2 ** (-dt / half_life)
    left = [N0]
    n = N0
    for _ in range(steps):
        decays = rng.binomial(n, p)   # how many of the n survivors decay this step
        n -= decays
        left.append(n)
    return np.array(left)

half_life = 8.02          # iodine-131, days
dt = 0.25                 # days per step
t = np.arange(0, 40 + dt, dt)
sim = simulate(10_000, half_life, dt, len(t) - 1)
theory = 10_000 * 0.5 ** (t / half_life)

plt.figure(figsize=(7, 4))
plt.plot(t, sim, label="simulation (10,000 nuclei)")
plt.plot(t, theory, "--", label="N₀ (½)^(t / T½)")
for k in range(1, 5):
    plt.axvline(k * half_life, color="gray", lw=0.6, ls=":")
plt.xlabel("time (days)"); plt.ylabel("nuclei left")
plt.title("Iodine-131 decay")
plt.legend(); plt.show()
print(f"After one half-life: {sim[int(half_life / dt)]} left (theory 5,000)")

## 2. Small samples are noisy

Run the same isotope with 100 nuclei five times. How far do the runs stray from the theory?

In [ ]:
plt.figure(figsize=(7, 4))
for run in range(5):
    plt.plot(t, simulate(100, half_life, dt, len(t) - 1) / 100, alpha=0.7)
plt.plot(t, 0.5 ** (t / half_life), "k--", lw=2, label="theory")
plt.xlabel("time (days)"); plt.ylabel("fraction left")
plt.title("Five runs of 100 nuclei")
plt.legend(); plt.show()

for N0 in (100, 10_000, 1_000_000):
    worst = np.max(np.abs(simulate(N0, half_life, dt, len(t) - 1) / N0 - 0.5 ** (t / half_life)))
    print(f"N0 = {N0:>9,}: largest gap from theory = {100 * worst:.2f}%")

## 3. Measure the half-life from data

Taking the log of $N = N_0 e^{-\lambda t}$ gives a straight line:

$$\ln N = \ln N_0 - \lambda t$$

so the slope of $\ln N$ against $t$ is $-\lambda$. Fit it with `np.polyfit`, then $T_{1/2} = \ln 2 / \lambda$. Pretend you don't know the half-life of the 'mystery' data below.

In [ ]:
mystery_half_life = 4.5     # hours (hidden from the student in the website version)
dt_h = 0.1
t_h = np.arange(0, 15 + dt_h, dt_h)
data = simulate(10_000, mystery_half_life, dt_h, len(t_h) - 1)

ok = data > 0                                   # log(0) is undefined
slope, intercept = np.polyfit(t_h[ok], np.log(data[ok]), 1)
lam = -slope
print(f"decay constant λ = {lam:.4f} per hour")
print(f"half-life = ln 2 / λ = {np.log(2) / lam:.2f} hours")

plt.figure(figsize=(7, 4))
plt.plot(t_h[ok], np.log(data[ok]), ".", ms=3, label="ln N from data")
plt.plot(t_h, intercept + slope * t_h, label=f"fit: slope = −{lam:.3f}")
plt.xlabel("time (hours)"); plt.ylabel("ln N")
plt.legend(); plt.show()

## 4. Carbon dating

Carbon-14 has a half-life of 5,730 years. If a fraction $f$ is left, the number of half-lives passed is $\log_2(1/f)$.

In [ ]:
C14_HALF_LIFE = 5730  # years

def carbon_age(fraction_left):
    # Age in years of a sample with this fraction of its carbon-14 left
    return C14_HALF_LIFE * np.log2(1 / fraction_left)

for f in (0.5, 0.25, 0.1, 0.01):
    print(f"{100 * f:5.1f}% left  ->  {carbon_age(f):8,.0f} years old")

## Newton vs quantum

No Newtonian calculation can say when *your* nucleus will decay; `rng.binomial` is the honest model. Yet with a million nuclei the curve is as smooth as any classical law.

## Exercises

1. **[Core]** Iodine-131 starts at 800 nuclei. Predict how many remain after 24 days, then check with `simulate`.
2. **[Extension]** How old is a wooden beam with 70% of its carbon-14 left?
3. **[Extension]** Fit the half-life from the 100-nucleus runs. How much do the five answers differ? Why?
4. **[Challenge]** Activity is $A = \lambda N$. Plot the number of decays per step from your simulation and show it also halves every half-life.